In [ ]:
# Cell 1: Imports

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import time
from pathlib import Path
from tqdm import tqdm
import random
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("="*80)
print(" "*20 + "SWIN TRANSFORMER 3D TRAINING")
print("="*80)
print(f"PyTorch: {torch.__version__}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
print("="*80)

In [ ]:
# Cell 2: Configuration

class Config:
    # This notebook is in notebooks/ folder
    # Go UP one level (..) to get to project root
    PROJECT_ROOT = Path('..')
    
    # Data from patches/ folder (parallel to notebooks/)
    DATA_DIR = PROJECT_ROOT / 'patches'
    
    # Outputs at project root level
    CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints'
    RESULTS_DIR = PROJECT_ROOT / 'results'
    PLOTS_DIR = PROJECT_ROOT / 'plots'
    
    # Model
    EMBED_DIM = 48
    DEPTHS = [2, 2, 2, 2]
    NUM_HEADS = [3, 6, 12, 24]
    WINDOW_SIZE = (4, 4, 4)
    PATCH_SIZE = 4
    INPUT_CHANNELS = 4
    NUM_CLASSES = 4
    
    # Training
    BATCH_SIZE = 2
    NUM_EPOCHS = 40
    LEARNING_RATE = 3e-4
    WEIGHT_DECAY = 1e-5
    WARMUP_EPOCHS = 5
    
    # Loss (WHAT WORKED!)
    CLASS_WEIGHTS = [1.0, 15.0, 8.0, 15.0]
    FOCAL_GAMMA = 2.0
    DICE_WEIGHT = 0.7
    FOCAL_WEIGHT = 0.3
    
    # Augmentation
    USE_AUGMENTATION = True
    FLIP_PROB = 0.5
    ROTATE_PROB = 0.5
    INTENSITY_SHIFT_PROB = 0.5
    
    # System
    USE_AMP = True
    GRADIENT_CLIP = 1.0
    NUM_WORKERS = 0

config = Config()

# Create output directories
for d in [config.CHECKPOINT_DIR, config.RESULTS_DIR, config.PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Verify data directory
print(f"\n🔍 PATH VERIFICATION:")
print(f"="*80)
print(f"Current directory: {Path.cwd()}")
print(f"Project root: {config.PROJECT_ROOT.resolve()}")
print(f"\nData directory: {config.DATA_DIR.resolve()}")
print(f"  Exists: {config.DATA_DIR.exists()}")

if config.DATA_DIR.exists():
    train_img = config.DATA_DIR / 'train' / 'images'
    train_msk = config.DATA_DIR / 'train' / 'masks'
    val_img = config.DATA_DIR / 'val' / 'images'
    val_msk = config.DATA_DIR / 'val' / 'masks'
    
    print(f"\n📁 Patch folders:")
    print(f"  Train images: {len(list(train_img.glob('*.npy')))} files")
    print(f"  Train masks:  {len(list(train_msk.glob('*.npy')))} files")
    print(f"  Val images:   {len(list(val_img.glob('*.npy')))} files")
    print(f"  Val masks:    {len(list(val_msk.glob('*.npy')))} files")
else:
    print(f"\n❌ ERROR: Data directory not found!")
    print(f"   Run 01_extract_patches.ipynb first!")

print(f"\n📋 CONFIGURATION:")
print(f"="*80)
print(f"  Epochs: {config.NUM_EPOCHS}")
print(f"  Batch size: {config.BATCH_SIZE}")
print(f"  Learning rate: {config.LEARNING_RATE}")
print(f"  Class weights: {config.CLASS_WEIGHTS}")
print(f"  Loss: {config.DICE_WEIGHT}× Dice + {config.FOCAL_WEIGHT}× Focal")
print("="*80)

In [ ]:
# Cell 3: Augmentation

class Augmentation3D:
    def __init__(self, flip_prob=0.5, rotate_prob=0.5, intensity_prob=0.5):
        self.flip_prob = flip_prob
        self.rotate_prob = rotate_prob
        self.intensity_prob = intensity_prob
    
    def random_flip(self, image, mask):
        if np.random.random() < self.flip_prob:
            axis = np.random.choice([0, 1, 2])
            image = torch.flip(image, dims=[axis + 1])
            mask = torch.flip(mask, dims=[axis])
        return image, mask
    
    def random_rotate_90(self, image, mask):
        if np.random.random() < self.rotate_prob:
            k = np.random.randint(1, 4)
            image = torch.rot90(image, k, dims=[2, 3])
            mask = torch.rot90(mask, k, dims=[1, 2])
        return image, mask
    
    def random_intensity_shift(self, image, mask):
        if np.random.random() < self.intensity_prob:
            shift = torch.randn(image.shape[0], 1, 1, 1) * 0.1
            image = image + shift
        return image, mask
    
    def __call__(self, image, mask):
        image, mask = self.random_flip(image, mask)
        image, mask = self.random_rotate_90(image, mask)
        image, mask = self.random_intensity_shift(image, mask)
        return image, mask

print("✓ Augmentation defined")

In [ ]:
# Cell 4: Dataset

class BraTSDataset(Dataset):
    def __init__(self, data_dir, split='train', augmentation=None):
        self.data_dir = data_dir
        self.split = split
        self.augmentation = augmentation
        
        self.image_dir = os.path.join(data_dir, split, 'images')
        self.mask_dir = os.path.join(data_dir, split, 'masks')
        
        self.image_files = sorted([f for f in os.listdir(self.image_dir) if f.endswith('.npy')])
        self.mask_files = sorted([f for f in os.listdir(self.mask_dir) if f.endswith('.npy')])
        
        assert len(self.image_files) == len(self.mask_files), \
            f"Mismatch: {len(self.image_files)} images vs {len(self.mask_files)} masks"
        
        assert len(self.image_files) > 0, f"No patches found in {self.image_dir}"
    
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        image = np.load(os.path.join(self.image_dir, self.image_files[idx]))
        mask = np.load(os.path.join(self.mask_dir, self.mask_files[idx]))
        
        image = torch.from_numpy(image).float()
        mask = torch.from_numpy(mask).long()
        mask[mask == 4] = 3
        
        if self.augmentation and self.split == 'train':
            image, mask = self.augmentation(image, mask)
        
        return image, mask

augmentation = Augmentation3D(
    flip_prob=config.FLIP_PROB,
    rotate_prob=config.ROTATE_PROB,
    intensity_prob=config.INTENSITY_SHIFT_PROB
) if config.USE_AUGMENTATION else None

print("\n" + "="*80)
print("LOADING DATASETS")
print("="*80)

try:
    train_dataset = BraTSDataset(config.DATA_DIR, split='train', augmentation=augmentation)
    val_dataset = BraTSDataset(config.DATA_DIR, split='val', augmentation=None)
    
    train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True, 
                             num_workers=config.NUM_WORKERS, drop_last=True, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False, 
                           num_workers=config.NUM_WORKERS, pin_memory=True)
    
    print(f"✓ Train: {len(train_dataset)} patches ({len(train_loader)} batches)")
    print(f"✓ Val: {len(val_dataset)} patches ({len(val_loader)} batches)")
    print(f"✓ Total: {len(train_dataset) + len(val_dataset)} patches")
    
    # Sample check
    sample_img, sample_mask = train_dataset[0]
    print(f"\nSample shapes:")
    print(f"  Image: {sample_img.shape} (expected: torch.Size([4, 64, 64, 64]))")
    print(f"  Mask: {sample_mask.shape} (expected: torch.Size([64, 64, 64]))")
    print("="*80)
    
except Exception as e:
    print(f"\n❌ ERROR loading datasets: {e}")
    print(f"\nMake sure you ran 01_extract_patches.ipynb first!")
    raise

In [ ]:
# Cell 5: Model (Swin Transformer 3D)

class PatchEmbed3D(nn.Module):
    def __init__(self, patch_size=4, in_chans=4, embed_dim=96):
        super().__init__()
        self.proj = nn.Conv3d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.norm = nn.LayerNorm(embed_dim)
    def forward(self, x):
        x = self.proj(x)
        B, C, D, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)
        x = self.norm(x).transpose(1, 2).view(B, C, D, H, W)
        return x

def window_partition(x, window_size):
    B, D, H, W, C = x.shape
    Wd, Wh, Ww = window_size
    pad_d, pad_h, pad_w = (Wd-D%Wd)%Wd, (Wh-H%Wh)%Wh, (Ww-W%Ww)%Ww
    if pad_d > 0 or pad_h > 0 or pad_w > 0:
        x = F.pad(x, (0, 0, 0, pad_w, 0, pad_h, 0, pad_d))
    B, D, H, W, C = x.shape
    x = x.view(B, D//Wd, Wd, H//Wh, Wh, W//Ww, Ww, C)
    return x.permute(0,1,3,5,2,4,6,7).contiguous().view(-1, Wd*Wh*Ww, C), (D,H,W)

def window_reverse(windows, window_size, original_size):
    Wd, Wh, Ww = window_size
    D, H, W = original_size
    C = windows.shape[-1]
    B = int(windows.shape[0] / (D*H*W / Wd / Wh / Ww))
    x = windows.view(B, D//Wd, H//Wh, W//Ww, Wd, Wh, Ww, C)
    return x.permute(0,1,4,2,5,3,6,7).contiguous().view(B, D, H, W, C)

class WindowAttention3D(nn.Module):
    def __init__(self, dim, window_size, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.scale = (dim // num_heads) ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
    def forward(self, x):
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C//self.num_heads).permute(2,0,3,1,4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = F.softmax((q @ k.transpose(-2,-1)) * self.scale, dim=-1)
        return self.proj((attn @ v).transpose(1,2).reshape(B_, N, C))

class SwinBlock3D(nn.Module):
    def __init__(self, dim, num_heads, window_size):
        super().__init__()
        self.window_size = window_size
        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention3D(dim, window_size, num_heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, dim*4), nn.GELU(), nn.Linear(dim*4, dim))
    def forward(self, x):
        B, D, H, W, C = x.shape
        shortcut = x
        x = self.norm1(x)
        x_w, ps = window_partition(x, self.window_size)
        x = window_reverse(self.attn(x_w), self.window_size, ps)[:,:D,:H,:W,:].contiguous()
        x = shortcut + x + self.mlp(self.norm2(x))
        return x

class SwinTransformer3D(nn.Module):
    def __init__(self, ic=4, nc=4, ed=48, dp=[2,2,2,2], nh=[3,6,12,24], ws=(4,4,4), ps=4):
        super().__init__()
        self.patch_embed = PatchEmbed3D(ps, ic, ed)
        self.layers = nn.ModuleList([nn.ModuleList([SwinBlock3D(int(ed*2**i),nh[i],ws) for _ in range(dp[i])]) for i in range(len(dp))])
        self.down = nn.ModuleList([nn.Conv3d(int(ed*2**i),int(ed*2**(i+1)),2,2) for i in range(len(dp)-1)])
        self.up = nn.ModuleList([nn.ConvTranspose3d(int(ed*2**i),int(ed*2**(i-1)),2,2) for i in range(len(dp)-1,0,-1)])
        self.final_up = nn.ConvTranspose3d(ed, ed, ps, ps)
        self.head = nn.Conv3d(ed, nc, 1)
    def forward(self, x):
        B,C,D,H,W = x.shape
        x = self.patch_embed(x)
        for i, blocks in enumerate(self.layers):
            x = x.permute(0,2,3,4,1).contiguous()
            for b in blocks: x = b(x)
            x = x.permute(0,4,1,2,3).contiguous()
            if i < len(self.layers)-1: x = self.down[i](x)
        for up in self.up: x = up(x)
        x = self.final_up(x)
        if x.shape[2:] != (D,H,W): x = F.interpolate(x, size=(D,H,W), mode='trilinear', align_corners=False)
        return self.head(x)

model = SwinTransformer3D(
    config.INPUT_CHANNELS, config.NUM_CLASSES, config.EMBED_DIM, 
    config.DEPTHS, config.NUM_HEADS, config.WINDOW_SIZE, config.PATCH_SIZE
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"\n✓ Model loaded: {total_params:,} parameters")

In [ ]:
# Cell 6: Loss & Optimizer

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma
    def forward(self, pred, target):
        ce = F.cross_entropy(pred, target, reduction='none')
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()

class DiceLoss(nn.Module):
    def __init__(self, weight=None, smooth=1e-5):
        super().__init__()
        self.weight = weight or [1,1,1,1]
        self.smooth = smooth
    def forward(self, pred, target):
        pred = F.softmax(pred, dim=1)
        target_oh = F.one_hot(target, pred.shape[1]).permute(0,4,1,2,3).float()
        loss = 0
        for c in range(pred.shape[1]):
            inter = (pred[:,c]*target_oh[:,c]).sum()
            union = pred[:,c].sum() + target_oh[:,c].sum()
            loss += self.weight[c] * (1 - (2*inter+self.smooth)/(union+self.smooth))
        return loss / sum(self.weight)

focal_loss = FocalLoss(gamma=config.FOCAL_GAMMA)
dice_loss = DiceLoss(weight=config.CLASS_WEIGHTS)
criterion = lambda p,t: config.DICE_WEIGHT * dice_loss(p,t) + config.FOCAL_WEIGHT * focal_loss(p,t)

def compute_dice(pred, target):
    et = (pred==3).float()
    tc = ((pred==1)|(pred==3)).float()
    wt = ((pred==1)|(pred==2)|(pred==3)).float()
    t_et = (target==3).float()
    t_tc = ((target==1)|(target==3)).float()
    t_wt = ((target==1)|(target==2)|(target==3)).float()
    return (
        (2*(et*t_et).sum()/(et.sum()+t_et.sum()+1e-8)).item(),
        (2*(tc*t_tc).sum()/(tc.sum()+t_tc.sum()+1e-8)).item(),
        (2*(wt*t_wt).sum()/(wt.sum()+t_wt.sum()+1e-8)).item()
    )

optimizer = torch.optim.AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.NUM_EPOCHS, eta_min=1e-6)
scaler = GradScaler(enabled=config.USE_AMP)

print("✓ Loss functions and optimizer configured")

In [ ]:
# Cell 7: Training Loop

print("\n" + "="*80)
print("STARTING TRAINING")
print("="*80)
print(f"Epochs: {config.NUM_EPOCHS}")
print(f"Estimated time: ~{config.NUM_EPOCHS * 15 / 60:.1f} hours")
print("="*80 + "\n")

best_dice = 0.0
history = {'train_loss':[], 'val_loss':[], 'dice_et':[], 'dice_tc':[], 'dice_wt':[], 'lr':[]}

for epoch in range(1, config.NUM_EPOCHS+1):
    start = time.time()
    
    # Warmup
    if epoch <= config.WARMUP_EPOCHS:
        lr_scale = epoch / config.WARMUP_EPOCHS
        for pg in optimizer.param_groups:
            pg['lr'] = config.LEARNING_RATE * lr_scale
    
    # Train
    model.train()
    train_loss = 0
    for imgs, msks in tqdm(train_loader, desc=f"Epoch {epoch}/{config.NUM_EPOCHS} [Train]", leave=False):
        imgs, msks = imgs.to(device), msks.to(device)
        optimizer.zero_grad()
        with autocast(enabled=config.USE_AMP):
            loss = criterion(model(imgs), msks)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRADIENT_CLIP)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()
    train_loss /= len(train_loader)
    
    # Validate
    model.eval()
    val_loss = 0
    dice_scores = {'et':[], 'tc':[], 'wt':[]}
    with torch.no_grad():
        for imgs, msks in tqdm(val_loader, desc=f"Epoch {epoch}/{config.NUM_EPOCHS} [Val]", leave=False):
            imgs, msks = imgs.to(device), msks.to(device)
            with autocast(enabled=config.USE_AMP):
                outs = model(imgs)
                val_loss += criterion(outs, msks).item()
            preds = torch.argmax(outs, dim=1)
            for i in range(len(imgs)):
                d_et, d_tc, d_wt = compute_dice(preds[i], msks[i])
                dice_scores['et'].append(d_et)
                dice_scores['tc'].append(d_tc)
                dice_scores['wt'].append(d_wt)
    val_loss /= len(val_loader)
    
    dice_et = np.mean(dice_scores['et'])
    dice_tc = np.mean(dice_scores['tc'])
    dice_wt = np.mean(dice_scores['wt'])
    mean_dice = np.mean([dice_et, dice_tc, dice_wt])
    
    if epoch > config.WARMUP_EPOCHS:
        scheduler.step()
    lr = optimizer.param_groups[0]['lr']
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['dice_et'].append(dice_et)
    history['dice_tc'].append(dice_tc)
    history['dice_wt'].append(dice_wt)
    history['lr'].append(lr)
    
    elapsed = time.time() - start
    print(f"Epoch {epoch:2d}/{config.NUM_EPOCHS} | {elapsed:.1f}s | Loss: T={train_loss:.4f} V={val_loss:.4f} | Dice: ET={dice_et:.3f} TC={dice_tc:.3f} WT={dice_wt:.3f} Mean={mean_dice:.3f} | LR={lr:.2e}", end="")
    
    if mean_dice > best_dice:
        best_dice = mean_dice
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'dice_et': dice_et,
            'dice_tc': dice_tc,
            'dice_wt': dice_wt,
            'mean_dice': mean_dice
        }, config.CHECKPOINT_DIR / 'best_model.pth')
        print(f" 🏆 BEST")
    else:
        print()
    
    if epoch % 10 == 0:
        torch.save(model.state_dict(), config.CHECKPOINT_DIR / f'epoch_{epoch}.pth')

print(f"\n" + "="*80)
print(f"✅ TRAINING COMPLETE!")
print(f"Best Dice: {best_dice:.4f}")
print("="*80)

In [ ]:
# Cell 8: Results

pd.DataFrame(history).to_csv(config.RESULTS_DIR / 'history.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(history['train_loss'])+1)
axes[0].plot(epochs, history['train_loss'], 'b-', linewidth=2, label='Train')
axes[0].plot(epochs, history['val_loss'], 'r-', linewidth=2, label='Val')
axes[0].set_xlabel('Epoch', fontweight='bold')
axes[0].set_ylabel('Loss', fontweight='bold')
axes[0].set_title('Loss Curves', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs, history['dice_et'], 'r-', linewidth=2, label='ET')
axes[1].plot(epochs, history['dice_tc'], 'g-', linewidth=2, label='TC')
axes[1].plot(epochs, history['dice_wt'], 'b-', linewidth=2, label='WT')
axes[1].axhline(best_dice, color='orange', linestyle='--', linewidth=2, label=f'Best: {best_dice:.3f}')
axes[1].set_xlabel('Epoch', fontweight='bold')
axes[1].set_ylabel('Dice Score', fontweight='bold')
axes[1].set_title('Dice Scores', fontweight='bold')
axes[1].set_ylim([0, 1])
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(config.PLOTS_DIR / 'training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n📊 Final Results:")
print(f"  Best Mean Dice: {best_dice:.4f}")
print(f"  Best ET: {max(history['dice_et']):.4f}")
print(f"  Best TC: {max(history['dice_tc']):.4f}")
print(f"  Best WT: {max(history['dice_wt']):.4f}")
print(f"\n📁 Saved:")
print(f"  Model: {config.CHECKPOINT_DIR / 'best_model.pth'}")
print(f"  History: {config.RESULTS_DIR / 'history.csv'}")
print(f"  Plot: {config.PLOTS_DIR / 'training_curves.png'}")